In [ ]:
import os


if os.getcwd().endswith("notebooks"):
    os.chdir("..")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from sklearn.linear_model import RANSACRegressor


from terramind_ad.zarr import load_features, load_timeseries
from terramind_ad.detect.base import to_decimal_year, create_harmonic_features

%matplotlib inline

## 1. Load Data

In [ ]:
idx = 180
site_id = "greece_wildfire_2025"
event_date = "2025-08-12"

In [ ]:
# load PC1 features
features_path = Path(f"data/{site_id}/features/s2/features_pc1.zarr")
features, timestamps, metadata = load_features(features_path)
features = (features - features.min()) / (features.max() - features.min())
# load feature-level cloud mask
features_cm_path = Path(f"data/{site_id}/features/s2/features_cm.zarr")
features_cm = load_timeseries(features_cm_path)

print(f"Feature cloud mask shape: {features_cm.shape}")
print(f"Features shape: {features.shape}")  # (T, H, W, 1)
print(f"Timestamps: {len(timestamps)}")
print(f"Date range: {timestamps[0]} to {timestamps[-1]}")

In [ ]:
# load RGB images from zarr
images_zarr = Path(f"data/{site_id}/images/s2/timeseries.zarr")
images_da = load_timeseries(images_zarr)
print(f"Images shape: {images_da.shape}")

# load cloud masks
clouds_zarr = Path(f"data/{site_id}/images/s2/clouds.zarr")
clouds_da = load_timeseries(clouds_zarr)
print(f"Clouds shape: {clouds_da.shape}")

# get RGB for one timestep (index 10)
rgb_bands = images_da.isel(time=idx).isel(band=[3, 2, 1]).values  # R, G, B
rgb = np.clip(rgb_bands / 5000, 0, 1).transpose(1, 2, 0)  # (H, W, C)

# get cloud mask for same timestep
cloud_mask = clouds_da.isel(time=idx).values

print(f"RGB shape: {rgb.shape}")
print(f"Cloud mask shape: {cloud_mask.shape}")
print(f"Reference image: {images_da.time[idx].values}")

In [ ]:
feats_pc1 = features[idx,:,:,0]
feat_clouds = features_cm.isel(time=idx).values
print(f"feats shape: {feats_pc1.shape}")
print(f"cloud shape: {feat_clouds.shape}")

## 2. Select Pixel Coordinate

Plot RGB and features to select a coordinate to debug.

In [ ]:
# select coordinate
y, x = 80, 75

# apply cloud mask to RGB
rgb_masked = rgb.copy()
rgb_masked[cloud_mask >= 1] = [1, 0, 1]
feat_masked = feats_pc1.copy()
feat_masked[feat_clouds >= 1] = 0

# apply feature cloud mask to PC1
fig, axes = plt.subplots(1, 2, figsize=(12, 7))


axes[0].imshow(rgb_masked)
axes[0].plot(x * 16, y * 16, "r+", markersize=20, markeredgewidth=3)
axes[0].set_title(f"Selected pixel: ({y}, {x})")
axes[0].axis("off")

im = axes[1].imshow(feat_masked)
axes[1].plot(x, y, "r+", markersize=9, markeredgewidth=2)
axes[1].set_title(f"PC1 - pixel ({y}, {x})")
# axes[1].axis("off")
plt.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.show()

## 3. Extract Time Series at Selected Pixel

In [ ]:
# get PC1 time series at selected pixel
pc1_series = features[:, y, x, 0]  # (T,)
valid_mask = features_cm.values[:, y, x] < 1
pc1_clear = pc1_series[valid_mask]
timestamps_clear = np.array(timestamps)[np.argwhere(valid_mask).flatten()].tolist()

print(f"Time series length: {len(pc1_series)}")
print(f"Value range: [{pc1_series.min():.3f}, {pc1_series.max():.3f}]")

# convert timestamps to decimal years
timestamps_dec = to_decimal_year(timestamps)
timestamps_dec_clear = to_decimal_year(timestamps_clear)
print(f"Time range: {timestamps_dec.min():.2f} - {timestamps_dec.max():.2f}")

plt.scatter(timestamps_dec, pc1_series, marker=".", color="red")
plt.scatter(timestamps_dec_clear, pc1_clear, marker="o", color="blue")
plt.show()


## 6. RANSAC on Pre-Event Data

In [ ]:
# create harmonic features
period = 1.0  # annual cycle
x_features = create_harmonic_features(timestamps_dec_clear, period=period)
print(f"Harmonic features shape: {x_features.shape}")

In [ ]:
event_dt = datetime.strptime(event_date, "%Y-%m-%d")
timestamp_dts = [datetime.strptime(ts, "%Y-%m-%d") for ts in timestamps_clear]
event_idx = next((i for i, ts in enumerate(timestamp_dts) if ts >= event_dt), None)

print(f"Event index: {event_idx} / {len(timestamps_clear)}, date: {timestamps_clear[event_idx]}")

In [ ]:
pc1_pre = pc1_clear[:event_idx]
ransac_thresh_pre = (pc1_pre.max() - pc1_pre.min()) * 0.15
print(f"RANSAC threshold: {ransac_thresh_pre}")
ransac_pre = RANSACRegressor(residual_threshold=ransac_thresh_pre, random_state=42)
ransac_pre.fit(x_features[:event_idx], pc1_clear[:event_idx])

fitted_pre = ransac_pre.predict(x_features)
residuals_pre = np.abs(pc1_clear - fitted_pre)
inlier_mask_pre = ransac_pre.inlier_mask_
outlier_mask_pre = ~inlier_mask_pre

print(f"Training inliers: {inlier_mask_pre.sum()} / {event_idx}")
print(f"Training outliers: {outlier_mask_pre.sum()}")
print(f"Residual at event: {residuals_pre[event_idx]:.3f}")

In [ ]:
print("Fit on pre-event only:")
print(f"  Training outliers: {outlier_mask_pre.sum()} / {event_idx}")
print(f"  Event residual: {residuals_pre[event_idx]:.3f}")
# this is what we are interested in: POST-event anomalies, the rest we do not care.
print(f"  Post-event anomalies (residual > threshold): {(residuals_pre[event_idx:] > ransac_thresh_pre).sum()}")

In [ ]:
# cd stands for change detection
cd_map = residuals_pre > ransac_thresh_pre
# this is what we intend to showcase
cd_map_post = residuals_pre[event_idx:] > ransac_thresh_pre

In [ ]:
fig = plt.figure(figsize=(14, 4))

plt.scatter(timestamps_dec, pc1_series, marker="o", label="PC1", alpha=0.6, s=6)
plt.plot(timestamps_dec_clear, fitted_pre, "b-", label="Pre-event fit", linewidth=2)
plt.axvline(timestamps_dec[event_idx], color="green", linestyle="--", linewidth=2, label="Event")
plt.scatter(timestamps_dec_clear[event_idx:][cd_map_post], pc1_clear[event_idx:][cd_map_post], color="red", marker="x", label="Post-event anomalies")
plt.ylabel("PC1 value")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()